# OULAD — Weekly Student States

This notebook creates **one row per student–module–presentation–week** from the raw OULAD files.

## Main outputs

- `weekly_clicks`
- `cumulative_clicks`
- `weekly_active_days`
- `days_since_last_activity`
- `click_regularity`
- `cumulative_grade`
- `average_submission_delay`
- `weighted_assessment_progress`
- `due_weighted_progress`
- `missed_assessments`
- Optional targets such as `at_risk_target` and `withdraw_next_4_weeks`

## Temporal leakage rule

Each row for Week `W` uses only information that became available **on or before the end of Week W**.  
`final_result` and future withdrawal information must not be included among the predictor features.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import zipfile
import urllib.request
import gc

import numpy as np
import pandas as pd

# DuckDB reads the large studentVle.csv file more efficiently than loading it fully into memory.
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])

import duckdb

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [2]:
# =========================
# Configuration
# =========================

DATA_DIR = Path("data/oulad")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UCI_ZIP_URL = (
    "https://archive.ics.uci.edu/static/public/349/"
    "open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"
)
ZIP_PATH = DATA_DIR / "oulad.zip"

# Week 0 starts at day 0. Pre-course interactions with negative dates are excluded by default.
INCLUDE_PRECOURSE_ACTIVITY = False

# Prediction horizon for the future-withdrawal target.
WITHDRAWAL_HORIZON_DAYS = 28

KEYS = ["id_student", "code_module", "code_presentation"]
WEEK_KEYS = KEYS + ["week"]

In [3]:
def download_and_extract_oulad() -> None:
    # Download and extract the official UCI OULAD archive when files are absent.
    required = {
        "courses.csv",
        "studentInfo.csv",
        "studentRegistration.csv",
        "assessments.csv",
        "studentAssessment.csv",
        "vle.csv",
        "studentVle.csv",
    }

    existing = {p.name for p in DATA_DIR.rglob("*.csv")} if DATA_DIR.exists() else set()
    if required.issubset(existing):
        print("OULAD files already exist.")
        return

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading OULAD from the official UCI source...")
    urllib.request.urlretrieve(UCI_ZIP_URL, ZIP_PATH)

    print("Extracting the archive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

    existing = {p.name for p in DATA_DIR.rglob("*.csv")}
    missing = required - existing
    if missing:
        raise FileNotFoundError(f"Missing files after extraction: {sorted(missing)}")

    print("Download and extraction completed successfully.")


def csv_path(filename: str) -> Path:
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not locate {filename} under {DATA_DIR.resolve()}")
    return matches[0]


download_and_extract_oulad()

OULAD files already exist.


In [4]:
# =========================
# Load and clean the smaller OULAD tables
# =========================

courses = pd.read_csv(csv_path("courses.csv"))
student_info = pd.read_csv(csv_path("studentInfo.csv"))
registration = pd.read_csv(csv_path("studentRegistration.csv"))
assessments = pd.read_csv(csv_path("assessments.csv"))
student_assessment = pd.read_csv(csv_path("studentAssessment.csv"))
vle = pd.read_csv(csv_path("vle.csv"))

# Convert OULAD placeholder values such as "?" into real missing values.
for df in [
    courses,
    student_info,
    registration,
    assessments,
    student_assessment,
    vle,
]:
    df.replace(r"^\s*\?\s*$", np.nan, regex=True, inplace=True)

# Standardize identifiers.
student_info["id_student"] = pd.to_numeric(
    student_info["id_student"], errors="raise"
).astype("int64")
registration["id_student"] = pd.to_numeric(
    registration["id_student"], errors="raise"
).astype("int64")
student_assessment["id_student"] = pd.to_numeric(
    student_assessment["id_student"], errors="raise"
).astype("int64")

# Standardize temporal and assessment fields.
registration["date_registration"] = pd.to_numeric(
    registration["date_registration"], errors="coerce"
)
registration["date_unregistration"] = pd.to_numeric(
    registration["date_unregistration"], errors="coerce"
)
assessments["date"] = pd.to_numeric(
    assessments["date"], errors="coerce"
)
assessments["weight"] = pd.to_numeric(
    assessments["weight"], errors="coerce"
)
student_assessment["date_submitted"] = pd.to_numeric(
    student_assessment["date_submitted"], errors="coerce"
)
student_assessment["score"] = pd.to_numeric(
    student_assessment["score"], errors="coerce"
)
student_assessment["is_banked"] = (
    pd.to_numeric(student_assessment["is_banked"], errors="coerce")
    .fillna(0)
    .astype("int8")
)

print("Loaded tables:")
for name, df in {
    "courses": courses,
    "studentInfo": student_info,
    "studentRegistration": registration,
    "assessments": assessments,
    "studentAssessment": student_assessment,
    "vle": vle,
}.items():
    print(f"{name:22s} {df.shape}")

print(
    "\nMissing imd_band values:",
    int(student_info["imd_band"].isna().sum()),
)

Loaded tables:
courses                (22, 3)
studentInfo            (32593, 12)
studentRegistration    (32593, 5)
assessments            (206, 6)
studentAssessment      (173912, 5)
vle                    (6364, 6)

Missing imd_band values: 1111


In [5]:
# Audit missing values in the source tables.
missing_report_rows = []

for table_name, df in {
    "courses": courses,
    "studentInfo": student_info,
    "studentRegistration": registration,
    "assessments": assessments,
    "studentAssessment": student_assessment,
    "vle": vle,
}.items():
    for col in df.columns:
        count = int(df[col].isna().sum())
        if count:
            missing_report_rows.append({
                "table": table_name,
                "field": col,
                "missing_count": count,
                "missing_percent": round(100 * count / len(df), 4),
            })

if missing_report_rows:
    missing_report = (
        pd.DataFrame(missing_report_rows)
        .sort_values(
            ["missing_percent", "table", "field"],
            ascending=[False, True, True],
        )
        .reset_index(drop=True)
    )
    display(missing_report)
else:
    missing_report = pd.DataFrame(
        columns=["table", "field", "missing_count", "missing_percent"]
    )
    print("No missing values found in the inspected source tables.")

,table,field,missing_count,missing_percent
0,vle,week_from,5243,82.3853
1,vle,week_to,5243,82.3853
2,studentRegistration,date_unregistration,22521,69.0977
3,assessments,date,11,5.3398
4,studentInfo,imd_band,1111,3.4087
5,studentRegistration,date_registration,45,0.1381
6,studentAssessment,score,173,0.0995


## 1. Create the weekly-state skeleton

The skeleton respects both registration and withdrawal dates.

- A student is not given a weekly state before their recorded registration date.
- A weekly state ends at the earliest of the normal week end, the presentation end, or the day before withdrawal.
- Students who withdrew before Day 0 are excluded because they have no observable in-course week.

In [6]:
if "module_presentation_length" not in courses.columns:
    if "length" in courses.columns:
        courses = courses.rename(
            columns={"length": "module_presentation_length"}
        )
    else:
        raise KeyError(
            "courses.csv must contain "
            "'module_presentation_length' or 'length'."
        )

courses["module_presentation_length"] = pd.to_numeric(
    courses["module_presentation_length"], errors="raise"
).astype("int32")

base = (
    student_info
    .merge(
        courses[
            [
                "code_module",
                "code_presentation",
                "module_presentation_length",
            ]
        ],
        on=["code_module", "code_presentation"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        registration[
            [
                "id_student",
                "code_module",
                "code_presentation",
                "date_registration",
                "date_unregistration",
            ]
        ],
        on=KEYS,
        how="left",
        validate="one_to_one",
    )
)

def eligible_weeks(row: pd.Series) -> list[int]:
    course_end_day = int(row["module_presentation_length"]) - 1

    if pd.notna(row["date_registration"]):
        first_observable_day = max(0, int(row["date_registration"]))
    else:
        # Missing registration dates are retained and assumed observable
        # from course Day 0. A missingness indicator can be used later.
        first_observable_day = 0

    if pd.notna(row["date_unregistration"]):
        last_observable_day = min(
            course_end_day,
            int(row["date_unregistration"]) - 1,
        )
    else:
        last_observable_day = course_end_day

    if last_observable_day < first_observable_day:
        return []

    first_week = first_observable_day // 7
    last_week = last_observable_day // 7
    return list(range(first_week, last_week + 1))


base["week"] = base.apply(eligible_weeks, axis=1)

weekly = base.explode("week", ignore_index=True)
weekly = weekly.loc[weekly["week"].notna()].copy()
weekly["week"] = weekly["week"].astype("int16")

weekly["week_start_day"] = (
    weekly["week"].astype("int32") * 7
)

registration_start = (
    weekly["date_registration"]
    .fillna(0)
    .clip(lower=0)
    .astype("int32")
)

weekly["observation_start_day"] = np.maximum(
    weekly["week_start_day"],
    registration_start,
).astype("int32")

weekly["cutoff_day"] = (
    (weekly["week"].astype("int32") + 1) * 7 - 1
)

weekly["cutoff_day"] = np.minimum(
    weekly["cutoff_day"],
    weekly["module_presentation_length"].astype("int32") - 1,
)

withdrawal_mask = weekly["date_unregistration"].notna()
weekly.loc[withdrawal_mask, "cutoff_day"] = np.minimum(
    weekly.loc[withdrawal_mask, "cutoff_day"],
    (
        weekly.loc[withdrawal_mask, "date_unregistration"]
        .astype("int32")
        - 1
    ),
)

# Defensive filter for impossible or empty observation intervals.
weekly = weekly.loc[
    weekly["observation_start_day"] <= weekly["cutoff_day"]
].copy()

weekly["registration_date_missing"] = (
    weekly["date_registration"].isna().astype("int8")
)

weekly = weekly.sort_values(WEEK_KEYS).reset_index(drop=True)

print(f"Weekly skeleton rows: {len(weekly):,}")
display(
    weekly[
        WEEK_KEYS
        + [
            "date_registration",
            "date_unregistration",
            "observation_start_day",
            "cutoff_day",
            "final_result",
        ]
    ].head()
)

Weekly skeleton rows: 927,471


,id_student,code_module,code_presentation,week,date_registration,date_unregistration,observation_start_day,cutoff_day,final_result
0,6516,AAA,2014J,0,-52.0,NaN,0,6,Pass
1,6516,AAA,2014J,1,-52.0,NaN,7,13,Pass
2,6516,AAA,2014J,2,-52.0,NaN,14,20,Pass
3,6516,AAA,2014J,3,-52.0,NaN,21,27,Pass
4,6516,AAA,2014J,4,-52.0,NaN,28,34,Pass


## 2. Derive engagement features from `studentVle.csv`

This version joins daily VLE activity to each weekly snapshot using the snapshot's **exact observation interval**:

```text
observation_start_day <= activity date <= cutoff_day
```

This removes activity that occurred after a partial final-week cutoff and prevents states from using interactions recorded before registration.

In [7]:
student_vle_path = csv_path("studentVle.csv").resolve()
student_vle_sql_path = student_vle_path.as_posix().replace("'", "''")

weekly_cutoffs = weekly[
    WEEK_KEYS + ["observation_start_day", "cutoff_day"]
].copy()

con = duckdb.connect()
con.register("weekly_cutoffs", weekly_cutoffs)

date_condition = "v.date >= 0" if not INCLUDE_PRECOURSE_ACTIVITY else "TRUE"

query = f"""
WITH daily_activity AS (
    SELECT
        CAST(id_student AS BIGINT) AS id_student,
        code_module,
        code_presentation,
        CAST(date AS INTEGER) AS activity_day,
        SUM(sum_click) AS daily_clicks
    FROM read_csv_auto(
        '{student_vle_sql_path}',
        header=true
    ) AS v
    WHERE {date_condition}
    GROUP BY
        id_student,
        code_module,
        code_presentation,
        date
)
SELECT
    w.id_student,
    w.code_module,
    w.code_presentation,
    w.week,
    COALESCE(SUM(d.daily_clicks), 0) AS weekly_clicks,
    COUNT(d.activity_day) AS weekly_active_days,
    MAX(d.activity_day) AS last_activity_day,
    AVG(d.daily_clicks) AS mean_clicks_per_active_day,
    STDDEV_POP(d.daily_clicks) AS std_clicks_per_active_day
FROM weekly_cutoffs AS w
LEFT JOIN daily_activity AS d
    ON d.id_student = w.id_student
    AND d.code_module = w.code_module
    AND d.code_presentation = w.code_presentation
    AND d.activity_day BETWEEN
        w.observation_start_day AND w.cutoff_day
GROUP BY
    w.id_student,
    w.code_module,
    w.code_presentation,
    w.week
"""

vle_weekly = con.execute(query).df()
con.unregister("weekly_cutoffs")
con.close()

vle_weekly["week"] = vle_weekly["week"].astype("int16")

weekly = weekly.merge(
    vle_weekly,
    on=WEEK_KEYS,
    how="left",
    validate="one_to_one",
)

weekly["weekly_clicks"] = (
    weekly["weekly_clicks"].fillna(0).astype("int64")
)
weekly["weekly_active_days"] = (
    weekly["weekly_active_days"].fillna(0).astype("int16")
)

weekly = weekly.sort_values(WEEK_KEYS).reset_index(drop=True)
g = weekly.groupby(KEYS, sort=False)

weekly["observable_days_this_week"] = (
    weekly["cutoff_day"] - weekly["observation_start_day"] + 1
).astype("int16")

weekly["cumulative_clicks"] = g["weekly_clicks"].cumsum()
weekly["cumulative_active_days"] = g["weekly_active_days"].cumsum()
weekly["cumulative_observable_days"] = (
    g["observable_days_this_week"].cumsum()
)

weekly["last_activity_day_seen"] = (
    g["last_activity_day"].ffill()
)
weekly["never_active"] = (
    weekly["last_activity_day_seen"].isna().astype("int8")
)

weekly["days_since_last_activity"] = (
    weekly["cutoff_day"] - weekly["last_activity_day_seen"]
)

weekly.loc[
    weekly["last_activity_day_seen"].isna(),
    "days_since_last_activity",
] = (
    weekly.loc[
        weekly["last_activity_day_seen"].isna(),
        "cutoff_day",
    ]
    - weekly.loc[
        weekly["last_activity_day_seen"].isna(),
        "observation_start_day",
    ]
    + 1
)

weekly["days_since_last_activity"] = (
    weekly["days_since_last_activity"]
    .clip(lower=0)
    .astype("int32")
)

weekly["weekly_active_ratio"] = (
    weekly["weekly_active_days"]
    / weekly["observable_days_this_week"].replace(0, np.nan)
).fillna(0).clip(0, 1)

weekly["cumulative_active_day_ratio"] = (
    weekly["cumulative_active_days"]
    / weekly["cumulative_observable_days"].replace(0, np.nan)
).fillna(0).clip(0, 1)

# Click regularity is defined as 1 / (1 + coefficient of variation)
# across active days within the week. A separate active-day ratio
# captures how many days were active.
mean_clicks = weekly["mean_clicks_per_active_day"]
std_clicks = weekly["std_clicks_per_active_day"].fillna(0)

weekly["click_regularity"] = np.where(
    mean_clicks.gt(0),
    1.0 / (1.0 + std_clicks / mean_clicks),
    0.0,
)
weekly["click_regularity"] = (
    pd.Series(weekly["click_regularity"], index=weekly.index)
    .fillna(0)
    .clip(0, 1)
)

weekly["engagement_change"] = (
    g["weekly_clicks"].diff().fillna(0)
)

display(
    weekly[
        WEEK_KEYS
        + [
            "observation_start_day",
            "cutoff_day",
            "weekly_clicks",
            "cumulative_clicks",
            "weekly_active_days",
            "days_since_last_activity",
            "weekly_active_ratio",
            "cumulative_active_day_ratio",
            "click_regularity",
            "engagement_change",
        ]
    ].head(20)
)

,id_student,code_module,code_presentation,week,observation_start_day,cutoff_day,weekly_clicks,cumulative_clicks,weekly_active_days,days_since_last_activity,weekly_active_ratio,cumulative_active_day_ratio,click_regularity,engagement_change
0,6516,AAA,2014J,0,0,6,229,229,6,0,0.857143,0.857143,0.633997,0.0
1,6516,AAA,2014J,1,7,13,42,271,4,0,0.571429,0.714286,0.596973,-187.0
2,6516,AAA,2014J,2,14,20,79,350,5,0,0.714286,0.714286,0.567192,37.0
3,6516,AAA,2014J,3,21,27,193,543,3,0,0.428571,0.642857,0.527368,114.0
4,6516,AAA,2014J,4,28,34,69,612,4,0,0.571429,0.628571,0.561356,-124.0
5,6516,AAA,2014J,5,35,41,34,646,5,0,0.714286,0.642857,0.689696,-35.0
6,6516,AAA,2014J,6,42,48,10,656,2,4,0.285714,0.591837,0.714286,-24.0
7,6516,AAA,2014J,7,49,55,93,749,7,0,1.000000,0.642857,0.604832,83.0
8,6516,AAA,2014J,8,56,62,57,806,5,0,0.714286,0.650794,0.560601,-36.0
9,6516,AAA,2014J,9,63,69,61,867,5,0,0.714286,0.657143,0.550843,4.0


## 3. Derive academic features

The calculations are performed one module presentation at a time to reduce memory use.

- `cumulative_grade`: weighted mean of scores available by the cutoff.
- `average_submission_delay`: timing for current-attempt submissions only.
- `banked_assessment_count`: assessments transferred from a previous attempt.
- `weighted_assessment_progress`: completed assessment weight divided by total course assessment weight.
- `due_weighted_progress`: completed weight among assessments already due.
- `missed_assessments`: assessments that were due but not completed by the cutoff.

### Score-availability assumption

OULAD does not provide grade-release timestamps. This notebook therefore assumes that a recorded score is observable from the recorded submission date. This assumption must be reported as a limitation.

In [8]:
assessments_enriched = assessments.merge(
    courses[
        [
            "code_module",
            "code_presentation",
            "module_presentation_length",
        ]
    ],
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one",
)

assessments_enriched["assessment_date"] = pd.to_numeric(
    assessments_enriched["date"], errors="coerce"
)

missing_assessment_date = (
    assessments_enriched["assessment_date"].isna()
)
exam_mask = (
    assessments_enriched["assessment_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("exam")
)

# OULAD's missing assessment dates are generally final exams.
# Fill only missing exam dates with the presentation length.
assessments_enriched.loc[
    missing_assessment_date & exam_mask,
    "assessment_date",
] = assessments_enriched.loc[
    missing_assessment_date & exam_mask,
    "module_presentation_length",
]

remaining_missing_dates = int(
    assessments_enriched["assessment_date"].isna().sum()
)
if remaining_missing_dates:
    print(
        "Warning:",
        remaining_missing_dates,
        "non-exam assessment dates remain missing and will not "
        "be treated as due in timing calculations.",
    )

student_assessment_enriched = student_assessment.merge(
    assessments_enriched[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_date",
            "weight",
            "assessment_type",
        ]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one",
)

def build_assessment_features_for_presentation(
    snapshots: pd.DataFrame,
    students: pd.DataFrame,
    assessment_schedule: pd.DataFrame,
    student_results: pd.DataFrame,
) -> pd.DataFrame:
    if assessment_schedule.empty or snapshots.empty:
        return snapshots[["id_student", "week"]].assign(
            assessments_due_so_far=0,
            assessments_submitted_so_far=0,
            missed_assessments=0,
            late_submission_count=0,
            banked_assessment_count=0,
            average_submission_delay=np.nan,
            cumulative_grade=np.nan,
            weighted_assessment_progress=0.0,
            due_weighted_progress=np.nan,
        )

    plan = (
        students[["id_student"]]
        .drop_duplicates()
        .assign(_join_key=1)
        .merge(
            assessment_schedule[
                [
                    "id_assessment",
                    "assessment_date",
                    "weight",
                    "assessment_type",
                ]
            ].assign(_join_key=1),
            on="_join_key",
            how="inner",
        )
        .drop(columns="_join_key")
    )

    plan = plan.merge(
        student_results[
            [
                "id_student",
                "id_assessment",
                "date_submitted",
                "is_banked",
                "score",
            ]
        ],
        on=["id_student", "id_assessment"],
        how="left",
        validate="one_to_one",
    )

    total_weight = float(
        assessment_schedule["weight"].fillna(0).sum()
    )

    expanded = snapshots[
        ["id_student", "week", "cutoff_day"]
    ].merge(plan, on="id_student", how="left")

    due = (
        expanded["assessment_date"].notna()
        & expanded["assessment_date"].le(expanded["cutoff_day"])
    )

    submitted = (
        expanded["date_submitted"].notna()
        & expanded["date_submitted"].le(expanded["cutoff_day"])
    )

    banked = (
        expanded["is_banked"].fillna(0).eq(1)
    )

    # Banked scores may contribute to the academic state, but they do
    # not represent current-attempt submission timing.
    timing_submission = submitted & ~banked
    score_available = submitted & expanded["score"].notna()
    weight = expanded["weight"].fillna(0).astype(float)

    expanded["due_flag"] = due.astype("int8")
    expanded["submitted_flag"] = submitted.astype("int8")
    expanded["missed_flag"] = (due & ~submitted).astype("int8")
    expanded["banked_flag"] = (
        submitted & banked
    ).astype("int8")

    expanded["late_flag"] = (
        timing_submission
        & expanded["date_submitted"].gt(
            expanded["assessment_date"]
        )
    ).astype("int8")

    expanded["submission_delay_value"] = np.where(
        timing_submission,
        (
            expanded["date_submitted"]
            - expanded["assessment_date"]
        ),
        np.nan,
    )

    expanded["weighted_score_num"] = np.where(
        score_available,
        expanded["score"] * weight,
        0.0,
    )
    expanded["weighted_score_den"] = np.where(
        score_available,
        weight,
        0.0,
    )
    expanded["submitted_weight"] = np.where(
        submitted,
        weight,
        0.0,
    )
    expanded["due_weight"] = np.where(
        due,
        weight,
        0.0,
    )
    expanded["due_submitted_weight"] = np.where(
        due & submitted,
        weight,
        0.0,
    )

    agg = (
        expanded
        .groupby(["id_student", "week"], as_index=False)
        .agg(
            assessments_due_so_far=("due_flag", "sum"),
            assessments_submitted_so_far=(
                "submitted_flag",
                "sum",
            ),
            missed_assessments=("missed_flag", "sum"),
            late_submission_count=("late_flag", "sum"),
            banked_assessment_count=("banked_flag", "sum"),
            average_submission_delay=(
                "submission_delay_value",
                "mean",
            ),
            weighted_score_num=("weighted_score_num", "sum"),
            weighted_score_den=("weighted_score_den", "sum"),
            submitted_weight=("submitted_weight", "sum"),
            due_weight=("due_weight", "sum"),
            due_submitted_weight=(
                "due_submitted_weight",
                "sum",
            ),
        )
    )

    agg["cumulative_grade"] = (
        agg["weighted_score_num"]
        / agg["weighted_score_den"].replace(0, np.nan)
    )

    agg["weighted_assessment_progress"] = (
        agg["submitted_weight"] / total_weight
        if total_weight > 0
        else np.nan
    )

    agg["due_weighted_progress"] = (
        agg["due_submitted_weight"]
        / agg["due_weight"].replace(0, np.nan)
    )

    return agg.drop(
        columns=[
            "weighted_score_num",
            "weighted_score_den",
            "submitted_weight",
            "due_weight",
            "due_submitted_weight",
        ]
    )


assessment_feature_parts = []

presentations = (
    weekly[["code_module", "code_presentation"]]
    .drop_duplicates()
    .sort_values(["code_module", "code_presentation"])
)

for code_module, code_presentation in presentations.itertuples(
    index=False
):
    mask = (
        weekly["code_module"].eq(code_module)
        & weekly["code_presentation"].eq(code_presentation)
    )

    snapshots_p = weekly.loc[
        mask,
        ["id_student", "week", "cutoff_day"],
    ].copy()

    students_p = student_info.loc[
        student_info["code_module"].eq(code_module)
        & student_info["code_presentation"].eq(
            code_presentation
        ),
        ["id_student"],
    ].copy()

    schedule_p = assessments_enriched.loc[
        assessments_enriched["code_module"].eq(code_module)
        & assessments_enriched["code_presentation"].eq(
            code_presentation
        )
    ].copy()

    results_p = student_assessment_enriched.loc[
        student_assessment_enriched["code_module"].eq(
            code_module
        )
        & student_assessment_enriched[
            "code_presentation"
        ].eq(code_presentation)
    ].copy()

    features_p = build_assessment_features_for_presentation(
        snapshots=snapshots_p,
        students=students_p,
        assessment_schedule=schedule_p,
        student_results=results_p,
    )
    features_p["code_module"] = code_module
    features_p["code_presentation"] = code_presentation
    assessment_feature_parts.append(features_p)

    print(
        f"Processed {code_module}-{code_presentation}: "
        f"{len(snapshots_p):,} weekly states"
    )

    del (
        snapshots_p,
        students_p,
        schedule_p,
        results_p,
        features_p,
    )
    gc.collect()

assessment_features = pd.concat(
    assessment_feature_parts,
    ignore_index=True,
)

weekly = weekly.merge(
    assessment_features,
    on=WEEK_KEYS,
    how="left",
    validate="one_to_one",
)

Processed AAA-2013J: 13,549 weekly states
Processed AAA-2014J: 12,718 weekly states
Processed BBB-2013B: 48,979 weekly states
Processed BBB-2013J: 66,254 weekly states
Processed BBB-2014B: 40,930 weekly states
Processed BBB-2014J: 63,749 weekly states
Processed CCC-2014B: 44,011 weekly states
Processed CCC-2014J: 66,055 weekly states
Processed DDD-2013B: 35,450 weekly states
Processed DDD-2013J: 54,850 weekly states
Processed DDD-2014B: 31,295 weekly states
Processed DDD-2014J: 50,117 weekly states
Processed EEE-2013J: 33,629 weekly states
Processed EEE-2014B: 19,697 weekly states
Processed EEE-2014J: 36,884 weekly states
Processed FFF-2013B: 46,506 weekly states
Processed FFF-2013J: 69,059 weekly states
Processed FFF-2014B: 41,016 weekly states
Processed FFF-2014J: 65,527 weekly states
Processed GGG-2013J: 34,453 weekly states
Processed GGG-2014B: 26,959 weekly states
Processed GGG-2014J: 25,784 weekly states


## 4. Add targets, class-distribution reporting, and save Version 2

`final_result` is converted into a binary target:

- `1`: Fail or Withdrawn
- `0`: Pass or Distinction

The final-result field remains a **target source only** and must not enter the predictor matrix.

In [9]:
zero_fill_columns = [
    "assessments_due_so_far",
    "assessments_submitted_so_far",
    "missed_assessments",
    "late_submission_count",
    "banked_assessment_count",
    "weighted_assessment_progress",
]

for col in zero_fill_columns:
    if col not in weekly.columns:
        weekly[col] = np.nan
    weekly[col] = weekly[col].fillna(0)

for col in [
    "cumulative_grade",
    "average_submission_delay",
    "due_weighted_progress",
]:
    if col not in weekly.columns:
        weekly[col] = np.nan

weekly["at_risk_target"] = (
    weekly["final_result"].isin(["Fail", "Withdrawn"])
).astype("int8")

weekly["withdraw_next_4_weeks"] = (
    weekly["date_unregistration"].notna()
    & weekly["date_unregistration"].gt(
        weekly["cutoff_day"]
    )
    & weekly["date_unregistration"].le(
        weekly["cutoff_day"] + WITHDRAWAL_HORIZON_DAYS
    )
).astype("int8")

weekly["presentation_progress"] = (
    (weekly["cutoff_day"] + 1)
    / weekly["module_presentation_length"]
).clip(0, 1)

weekly = weekly.sort_values(WEEK_KEYS).reset_index(drop=True)

print("Final-result distribution at trajectory level:")
trajectory_results = (
    weekly.drop_duplicates(KEYS)["final_result"]
    .value_counts(dropna=False)
    .rename_axis("final_result")
    .to_frame("count")
)
trajectory_results["percent"] = (
    100 * trajectory_results["count"]
    / trajectory_results["count"].sum()
)
display(trajectory_results)

print("Weekly at-risk target distribution:")
weekly_risk_distribution = (
    weekly["at_risk_target"]
    .value_counts(dropna=False)
    .rename_axis("at_risk_target")
    .to_frame("count")
)
weekly_risk_distribution["percent"] = (
    100 * weekly_risk_distribution["count"]
    / weekly_risk_distribution["count"].sum()
)
display(weekly_risk_distribution)

print("Weekly four-week withdrawal target distribution:")
withdrawal_distribution = (
    weekly["withdraw_next_4_weeks"]
    .value_counts(dropna=False)
    .rename_axis("withdraw_next_4_weeks")
    .to_frame("count")
)
withdrawal_distribution["percent"] = (
    100 * withdrawal_distribution["count"]
    / withdrawal_distribution["count"].sum()
)
display(withdrawal_distribution)

parquet_path = (
    OUTPUT_DIR / "oulad_weekly_student_states_v2.parquet"
)
csv_path_out = (
    OUTPUT_DIR / "oulad_weekly_student_states_v2.csv.gz"
)

try:
    weekly.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path.resolve()}")
except ImportError:
    print(
        "Parquet engine is unavailable. "
        "Install pyarrow to save the Parquet file."
    )

weekly.to_csv(
    csv_path_out,
    index=False,
    compression="gzip",
)

print(f"Saved compressed CSV: {csv_path_out.resolve()}")
print(f"Final shape: {weekly.shape}")

Final-result distribution at trajectory level:


,count,percent
final_result,,
Pass,12361,41.907377
Withdrawn,7067,23.959181
Fail,7044,23.881204
Distinction,3024,10.252238


Weekly at-risk target distribution:


,count,percent
at_risk_target,,
0,574003,61.889051
1,353468,38.110949


Weekly four-week withdrawal target distribution:


,count,percent
withdraw_next_4_weeks,,
0,898043,96.827071
1,29428,3.172929


Saved Parquet: G:\Queen's\Graduation Project\Digital Twin Notebooks\outputs\oulad_weekly_student_states_v2.parquet
Saved compressed CSV: G:\Queen's\Graduation Project\Digital Twin Notebooks\outputs\oulad_weekly_student_states_v2.csv.gz
Final shape: (927471, 48)


In [10]:
# =========================
# Final validation checks
# =========================

assert not weekly.duplicated(WEEK_KEYS).any(), (
    "Duplicate weekly state detected."
)
assert weekly["week"].ge(0).all()
assert (
    weekly["observation_start_day"]
    <= weekly["cutoff_day"]
).all()

# No state may end before a positive registration date.
positive_registration = (
    weekly["date_registration"].notna()
    & weekly["date_registration"].gt(0)
)
assert (
    weekly.loc[positive_registration, "cutoff_day"]
    >= weekly.loc[positive_registration, "date_registration"]
).all(), "A weekly state was created before registration."

# No interaction may occur after the row's exact cutoff.
activity_rows = weekly["last_activity_day"].notna()
assert (
    weekly.loc[activity_rows, "last_activity_day"]
    <= weekly.loc[activity_rows, "cutoff_day"]
).all(), "Future VLE activity was included."

# No interaction may occur before the row's observation start.
assert (
    weekly.loc[activity_rows, "last_activity_day"]
    >= weekly.loc[activity_rows, "observation_start_day"]
).all()

withdrawal_rows = weekly["date_unregistration"].notna()
assert (
    weekly.loc[withdrawal_rows, "cutoff_day"]
    < weekly.loc[withdrawal_rows, "date_unregistration"]
).all(), "A state was created on or after withdrawal."

assert (
    weekly["weekly_active_days"]
    <= weekly["observable_days_this_week"]
).all()
assert weekly["weekly_active_ratio"].between(0, 1).all()
assert weekly["cumulative_active_day_ratio"].between(0, 1).all()
assert weekly["click_regularity"].between(0, 1).all()
assert weekly["presentation_progress"].between(0, 1).all()
assert weekly["cumulative_grade"].dropna().between(0, 100).all()

# Placeholder characters should not remain as category values.
if "imd_band" in weekly.columns:
    assert not weekly["imd_band"].astype(str).str.strip().eq("?").any()

print("All temporal, structural, and range checks passed.")

validation_summary = pd.Series({
    "weekly_rows": len(weekly),
    "columns": weekly.shape[1],
    "unique_students": weekly["id_student"].nunique(),
    "trajectories": weekly[KEYS].drop_duplicates().shape[0],
    "duplicate_weekly_keys": int(
        weekly.duplicated(WEEK_KEYS).sum()
    ),
    "activity_after_cutoff": int(
        (
            weekly["last_activity_day"].notna()
            & (
                weekly["last_activity_day"]
                > weekly["cutoff_day"]
            )
        ).sum()
    ),
    "states_before_registration": int(
        (
            positive_registration
            & (
                weekly["cutoff_day"]
                < weekly["date_registration"]
            )
        ).sum()
    ),
})
display(validation_summary.to_frame("value"))

display_columns = WEEK_KEYS + [
    "observation_start_day",
    "cutoff_day",
    "weekly_clicks",
    "cumulative_clicks",
    "weekly_active_days",
    "days_since_last_activity",
    "weekly_active_ratio",
    "cumulative_active_day_ratio",
    "click_regularity",
    "cumulative_grade",
    "missed_assessments",
    "banked_assessment_count",
    "average_submission_delay",
    "weighted_assessment_progress",
    "due_weighted_progress",
    "at_risk_target",
    "withdraw_next_4_weeks",
]

display(
    weekly[display_columns]
    .sort_values(
        ["id_student", "code_module", "code_presentation", "week"]
    )
    .head(50)
)

All temporal, structural, and range checks passed.


,value
weekly_rows,927471
columns,48
unique_students,26358
trajectories,29496
duplicate_weekly_keys,0
activity_after_cutoff,0
states_before_registration,0


,id_student,code_module,code_presentation,week,observation_start_day,cutoff_day,weekly_clicks,cumulative_clicks,weekly_active_days,days_since_last_activity,weekly_active_ratio,cumulative_active_day_ratio,click_regularity,cumulative_grade,missed_assessments,banked_assessment_count,average_submission_delay,weighted_assessment_progress,due_weighted_progress,at_risk_target,withdraw_next_4_weeks
0,6516,AAA,2014J,0,0,6,229,229,6,0,0.857143,0.857143,0.633997,NaN,0,0,NaN,0.0000,NaN,0,0
1,6516,AAA,2014J,1,7,13,42,271,4,0,0.571429,0.714286,0.596973,NaN,0,0,NaN,0.0000,NaN,0,0
2,6516,AAA,2014J,2,14,20,79,350,5,0,0.714286,0.714286,0.567192,60.000000,0,0,-2.0,0.0500,1.0,0,0
3,6516,AAA,2014J,3,21,27,193,543,3,0,0.428571,0.642857,0.527368,60.000000,0,0,-2.0,0.0500,1.0,0,0
4,6516,AAA,2014J,4,28,34,69,612,4,0,0.571429,0.628571,0.561356,60.000000,0,0,-2.0,0.0500,1.0,0,0
5,6516,AAA,2014J,5,35,41,34,646,5,0,0.714286,0.642857,0.689696,60.000000,0,0,-2.0,0.0500,1.0,0,0
6,6516,AAA,2014J,6,42,48,10,656,2,4,0.285714,0.591837,0.714286,60.000000,0,0,-2.0,0.0500,1.0,0,0
7,6516,AAA,2014J,7,49,55,93,749,7,0,1.000000,0.642857,0.604832,52.000000,0,0,-2.5,0.1500,1.0,0,0
8,6516,AAA,2014J,8,56,62,57,806,5,0,0.714286,0.650794,0.560601,52.000000,0,0,-2.5,0.1500,1.0,0,0
9,6516,AAA,2014J,9,63,69,61,867,5,0,0.714286,0.657143,0.550843,52.000000,0,0,-2.5,0.1500,1.0,0,0


## Predictor columns recommended for the baseline model

Do not use the following fields as predictors:

- `final_result`
- `date_unregistration`
- `at_risk_target`
- `withdraw_next_4_weeks`
- `id_student`

Use the Version 2 output in the Logistic Regression notebook:

```python
DATA_PATH = Path(
    "outputs/oulad_weekly_student_states_v2.parquet"
)
CSV_FALLBACK = Path(
    "outputs/oulad_weekly_student_states_v2.csv.gz"
)
```

Recommended prediction checkpoints for later evaluation are Weeks 3, 5, 8, and 10. Student-grouped splitting is suitable for an initial baseline, while presentation-aware holdout evaluation is required for testing transfer across module presentations.